# Latin Lemmatizer - Basic Usage Examples

This notebook demonstrates how to use the `latin_lemmatizer` package to query Latin lemmas and inflected forms via the API.

## Setup

**Before running this notebook**, make sure you have:
1. Run the installation cell below to install dependencies
2. Configure `LATIN_API_URL` and `LATIN_API_TOKEN` (see configuration cell below)
3. The API server is running and accessible

In [1]:
# Install dependencies if not already installed
import subprocess
import sys

packages = {
    "httpx": "httpx",
    "dotenv": "python-dotenv"
}

for module, package in packages.items():
    try:
        __import__(module)
        print(f"[OK] {package} already installed")
    except ImportError:
        print(f"Installing {package}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", package])
        print(f"[OK] {package} installed!")

[OK] httpx already installed
[OK] python-dotenv already installed


## Configure API Connection

You have two options to set `LATIN_API_URL` and `LATIN_API_TOKEN`:

**Option 1: Use a `.env` file (Recommended)**
- Create a `.env` file in the project root with:
  ```
  LATIN_API_URL=https://your-tunnel-url.trycloudflare.com
  LATIN_API_TOKEN=your-bearer-token
  ```
- The client will automatically load it when imported

**Option 2: Set it directly in this notebook**
- Uncomment and modify the cell below

In [2]:
import os

# Option 2: Set API credentials directly (uncomment and modify if not using .env file)
# os.environ["LATIN_API_URL"] = "https://your-tunnel-url.trycloudflare.com"
# os.environ["LATIN_API_TOKEN"] = "your-bearer-token"

# Check if API credentials are set
url = os.getenv("LATIN_API_URL")
token = os.getenv("LATIN_API_TOKEN")
if url and token:
    print(f"[OK] API configured: {url}")
else:
    missing = []
    if not url: missing.append("LATIN_API_URL")
    if not token: missing.append("LATIN_API_TOKEN")
    print(f"[WARNING] Missing: {', '.join(missing)}. Configure using Option 1 or 2 above.")

[OK] API configured: https://euros-leonard-route-employment.trycloudflare.com


## Import the Package

Now we can import the latin_lemmatizer package. It connects to the API using `LATIN_API_URL` and `LATIN_API_TOKEN`.


In [3]:
import sys
from pathlib import Path

# Add parent directory to path so we can import latin_lemmatizer
sys.path.insert(0, str(Path().resolve().parent))

from latin_lemmatizer import get_lemmas, get_forms, LatinLemmatizer

print("[OK] Successfully imported latin_lemmatizer")


[OK] Successfully imported latin_lemmatizer


## Example 1: Quick Lemma Lookup (top=True)

Use `get_lemmas(word, top=True)` for a quick single-result lookup — ideal for mass processing.
It returns the single best-matching lemma (or None).

In [4]:
# Quick lookup: top=True returns the single best match (or None)
words = ["amavi", "amat", "amo", "rosam", "rosa"]

for word in words:
    lemma = get_lemmas(word, top=True)
    if lemma:
        print(f"{word:15} -> {lemma['lemma_diac']:15} ({lemma['pos']})")
    else:
        print(f"{word:15} -> Not found")

amavi           -> ămo             (transitive verb I conjugation)
amat            -> ămo             (transitive verb I conjugation)
amo             -> ămo             (transitive verb I conjugation)
rosam           -> rŏsa            (feminine noun  I declension)
rosa            -> rŏsa            (feminine noun  I declension)


## Example 2: Disambiguation — All Matching Lemmas

When a form matches multiple lemmas, `get_lemmas()` (without `top=True`) returns all of them ranked by relevance. Use `pos=` to filter by part of speech.

In [5]:
# "rosam" matches both "rosa" (noun) and "rodo" (verb) — see all matches
all_matches = get_lemmas("rosam")
print(f"All lemmas matching 'rosam' ({len(all_matches)} results):")
for m in all_matches:
    print(f"  {m['lemma_diac']:15} ({m['pos']})")

# Use pos= filter to narrow down
print()
noun_only = get_lemmas("rosam", pos="noun")
print(f"Filtered to nouns: {noun_only[0]['lemma_diac']} ({noun_only[0]['pos']})" if noun_only else "No noun match")

All lemmas matching 'rosam' (3 results):
  rŏsa            (feminine noun  I declension)
  rōdo            (transitive verb  III conjugation)
  rosus           (adjective perfect participle  I class)

Filtered to nouns: rŏsa (feminine noun  I declension)


## Example 3: Get All Forms of a Lemma

Use `get_forms(lemma="...")` to retrieve all inflected forms of a specific lemma.

In [6]:
# Get all forms of "amo"
forms = get_forms(lemma="amo")
print(f"Found {len(forms)} forms of 'amo'\n")

# Show first 10 forms with their morphological features
print(f"{'Form':<20} {'Mood':<12} {'Tense':<15} {'Voice':<8} {'Person':<6} {'Number'}")
print("-" * 80)
for f in forms[:10]:
    print(f"{f['form_diac']:<20} {f['mood'] or '-':<12} {f['tense'] or '-':<15} {f['voice'] or '-':<8} {f['person'] or '-':<6} {f['number'] or '-'}")

if len(forms) > 10:
    print(f"\n... and {len(forms) - 10} more forms")

Found 229 forms of 'amo'

Form                 Mood         Tense           Voice    Person Number
--------------------------------------------------------------------------------
ămatōte              imperative   future          active   second plural
ămāto                imperative   future          active   second singular
ămanto               imperative   future          active   third  plural
ămāto                imperative   future          active   third  singular
ămātor               imperative   future          passive  second singular
ămantor              imperative   future          passive  third  plural
ămātor               imperative   future          passive  third  singular
ămāte                imperative   present         active   second plural
ăma                  imperative   present         active   second singular
ămamĭni              imperative   present         passive  second plural

... and 219 more forms


## Example 4: Filter Forms by Morphological Features

You can filter forms by mood, tense, voice, person, number, case, etc.

In [7]:
# Get present indicative active forms of "amo"
forms = get_forms(
    lemma="amo",
    mood="indicative",
    tense="present",
    voice="active"
)

print("Present Indicative Active Forms of 'amo':")
for f in forms:
    print(f"  {f['form_diac']:15} ({f['person']} {f['number']})")

Present Indicative Active Forms of 'amo':
  ămāmus          (first plural)
  ămo             (first singular)
  ămātis          (second plural)
  ămas            (second singular)
  ămant           (third plural)
  ămat            (third singular)


## Example 5: Inflect from an Existing Form

Instead of starting with a lemma, you can start with any inflected form and find other forms of the same lemma.

In [8]:
# Get plural forms from the same lemma as "amavi"
forms = get_forms(form="amavi", number="plural")

print(f"Found {len(forms)} plural forms from the same lemma as 'amavi':\n")
print(f"{'Form':<20} {'Tense':<15} {'Voice':<8} {'Person'}")
print("-" * 60)
for f in forms[:10]:
    print(f"{f['form_diac']:<20} {f['tense'] or '-':<15} {f['voice'] or '-':<8} {f['person'] or '-'}")

if len(forms) > 10:
    print(f"\n... and {len(forms) - 10} more")

Found 102 plural forms from the same lemma as 'amavi':

Form                 Tense           Voice    Person
------------------------------------------------------------
ămatōte              future          active   second
ămanto               future          active   third
ămantor              future          passive  third
ămāte                present         active   second
ămamĭni              present         passive  second
ămabĭmus             future          active   first
ămabĭtis             future          active   second
ămābunt              future          active   third
ămabĭmur             future          passive  first
ămabimĭni            future          passive  second

... and 92 more


## Example 6: Verb Forms (Infinitive, Participle, etc.)

Use the `verb_form` parameter to get specific verb forms like infinitive, participle, gerund, gerundive, or supine.

In [9]:
# Get infinitive forms of "amo"
forms = get_forms(lemma="amo", verb_form="infinitive")

print("Infinitive forms of 'amo':")
for f in forms:
    print(f"  {f['form_diac']:15} (tense: {f['tense']}, voice: {f['voice'] or '-'})")

Infinitive forms of 'amo':
  amata           (tense: future, voice: active)
  amatas          (tense: future, voice: active)
  amatūros        (tense: future, voice: active)
  amatam          (tense: future, voice: active)
  amatūm          (tense: future, voice: active)
  amatūrūm        (tense: future, voice: active)
  amatum          (tense: future, voice: passive)
  amavisse        (tense: perfect, voice: active)
  amata           (tense: perfect, voice: passive)
  amatae          (tense: perfect, voice: passive)
  amati           (tense: perfect, voice: passive)
  amata           (tense: perfect, voice: passive)
  amatum          (tense: perfect, voice: passive)
  amatus          (tense: perfect, voice: passive)
  ămāre           (tense: present, voice: active)
  ămāri           (tense: present, voice: passive)


## Example 7: Participles (Active vs Passive Entries)

The source dictionary treats active ("amo") and passive ("amor") as separate entries. Present/future participles are under the active lemma; perfect passive participles are under the passive lemma.

In [10]:
# Active participles of "amo"
forms = get_forms(lemma="amo", verb_form="participle")
print(f"Participles of 'amo' (active entry): {len(forms)} forms")
for f in forms:
    print(f"  {f['form_diac']:20} (tense: {f['tense']}, voice: {f['voice']})")

# Perfect passive participles are under the passive entry
print()
forms_passive = get_forms(lemma="amor", verb_form="participle")
print(f"Participles of 'amor' (passive entry): {len(forms_passive)} forms")
for f in forms_passive[:5]:
    print(f"  {f['form_diac']:20} (tense: {f['tense']}, voice: {f['voice']})")

Participles of 'amo' (active entry): 8 forms
  amata                (tense: future, voice: active)
  amatūm               (tense: future, voice: active)
  amatūrūs             (tense: future, voice: active)
  amata                (tense: perfect, voice: passive)
  amatum               (tense: perfect, voice: passive)
  amatus               (tense: perfect, voice: passive)
  ămans                (tense: present, voice: active)
  ămantis              (tense: present, voice: active)

Participles of 'amor' (passive entry): 8 forms
  amata                (tense: future, voice: active)
  amatūm               (tense: future, voice: active)
  amatūrūs             (tense: future, voice: active)
  amata                (tense: perfect, voice: passive)
  amatum               (tense: perfect, voice: passive)


## Example 8: Using the Client Class

For more control, use the `LatinLemmatizer` client class directly with a context manager.

In [11]:
# Using the client class with context manager
with LatinLemmatizer() as client:
    # Get best lemma match for "rosam"
    lemma = client.get_lemmas("rosam", top=True)
    print(f"Lemma of 'rosam': {lemma['lemma_diac'] if lemma else 'Not found'}")

    # Get all forms of "rosa" (noun declension)
    if lemma:
        forms = client.get_forms(lemma=lemma["lemma_nod"])
        print(f"\nFound {len(forms)} forms of '{lemma['lemma_diac']}':")
        print(f"{'Form':<15} {'Case':<12} {'Number':<10} {'Gender'}")
        print("-" * 50)
        for f in forms:
            print(f"{f['form_diac']:<15} {f['case'] or '-':<12} {f['number'] or '-':<10} {f['gender'] or '-'}")

Lemma of 'rosam': rŏsa

Found 12 forms of 'rŏsa':
Form            Case         Number     Gender
--------------------------------------------------
rosis           ablative     plural     feminine
rosas           accusative   plural     feminine
rosis           dative       plural     feminine
rosārum         genitive     plural     feminine
rosae           nominative   plural     feminine
rosae           vocative     plural     feminine
rosā            ablative     singular   feminine
rosam           accusative   singular   feminine
rosae           dative       singular   feminine
rosae           genitive     singular   feminine
rosă            nominative   singular   feminine
rosă            vocative     singular   feminine


## Tips

- `get_lemmas(word)` returns all matching lemmas ranked by relevance; add `top=True` for just the best one
- `get_lemmas(word, pos="noun")` filters by part of speech
- Always use `lemma` OR `form` (never both) in `get_forms()`
- All filter parameters are optional — omit them to get all matching forms
- `form_nod` = normalized text (no diacritics), `form_diac` = with diacritics
- Use the `LatinLemmatizer` client class for better connection management in long-running scripts
- The API URL changes each time you restart the Cloudflare Quick Tunnel — update your `.env` accordingly